# 03 — Feature Engineering
**Healthcare Operations Analytics**

Builds the feature set used later for the readmission model in `05_business_insights.ipynb`,
plus a few operational fields used across the SQL/dashboard layer.

In [1]:
import pandas as pd
import numpy as np

admissions = pd.read_csv('../data/cleaned/admissions_cleaned.csv', parse_dates=['admission_date','discharge_date'])
patients = pd.read_csv('../data/raw/patients.csv')
billing = pd.read_csv('../data/raw/billing.csv')
treatments = pd.read_csv('../data/raw/treatments.csv')
satisfaction = pd.read_csv('../data/raw/satisfaction_surveys.csv')

print(admissions.shape)

(46500, 13)


## Patient age at admission

In [2]:
patients['date_of_birth'] = pd.to_datetime(patients['date_of_birth'])
admissions = admissions.merge(patients[['patient_id','date_of_birth']], on='patient_id', how='left')
admissions['patient_age'] = ((admissions['admission_date'] - admissions['date_of_birth']).dt.days / 365.25).round(1)
admissions['age_group'] = pd.cut(admissions.patient_age, bins=[0,18,35,50,65,120],
                                   labels=['0-18','19-35','36-50','51-65','66+'])
admissions[['patient_age','age_group']].describe(include='all')

,patient_age,age_group
count,46500.000000,45444
unique,NaN,5
top,NaN,66+
freq,NaN,14190
mean,45.970946,NaN
std,28.052494,NaN
min,-3.400000,NaN
25%,21.500000,NaN
50%,45.700000,NaN
75%,70.500000,NaN


## Calendar features (weekend flag, month, is flu-season)

In [3]:
admissions['is_weekend_admission'] = admissions.admission_date.dt.dayofweek >= 5
admissions['admission_month'] = admissions.admission_date.dt.month
admissions['is_flu_season'] = admissions.admission_month.isin([12,1,2])
admissions[['is_weekend_admission','admission_month','is_flu_season']].head()

,is_weekend_admission,admission_month,is_flu_season
0,False,3,False
1,False,9,False
2,False,5,False
3,False,6,False
4,False,1,True


## Treatment intensity per admission

In [4]:
tx_agg = treatments.groupby('admission_id').agg(
    treatment_count=('treatment_id','count'),
    total_treatment_cost=('treatment_cost','sum')
).reset_index()
admissions = admissions.merge(tx_agg, on='admission_id', how='left')
admissions[['treatment_count','total_treatment_cost']] = admissions[['treatment_count','total_treatment_cost']].fillna(0)
admissions[['treatment_count','total_treatment_cost']].describe()

,treatment_count,total_treatment_cost
count,46500.000000,46500.000000
mean,1.700000,1623.974331
std,1.303879,1521.489701
min,0.000000,0.000000
25%,1.000000,406.960000
50%,2.000000,1287.510000
75%,2.000000,2448.867500
max,9.000000,12550.940000


## Financial features

In [5]:
billing_slim = billing[['admission_id','gross_charge','direct_operating_cost']].copy()
billing_slim['margin'] = billing_slim.gross_charge - billing_slim.direct_operating_cost
admissions = admissions.merge(billing_slim, on='admission_id', how='left')
admissions['cost_per_stay_day'] = (admissions.gross_charge / admissions.length_of_stay_days.replace(0, np.nan)).round(2)
admissions[['gross_charge','margin','cost_per_stay_day']].describe()

,gross_charge,margin,cost_per_stay_day
count,46500.000000,46500.000000,46500.000000
mean,10736.842191,4827.857633,2438.674671
std,9898.276284,4521.647808,2544.096622
min,38.470000,13.780000,8.740000
25%,5143.440000,2269.062500,1049.587500
50%,8718.460000,3880.280000,1838.575000
75%,13785.317500,6186.287500,3077.475000
max,389421.550000,163827.570000,76357.170000


## Satisfaction join (only ~58% of admissions have a response — kept as NaN, not imputed)

In [6]:
sat_slim = satisfaction[['admission_id','satisfaction_score']]
admissions = admissions.merge(sat_slim, on='admission_id', how='left')
print(f"Satisfaction response rate: {admissions.satisfaction_score.notna().mean()*100:.1f}%")

Satisfaction response rate: 58.0%


## Discharge efficiency bucket

Used directly in the readmission model as a categorical signal, since the earlier
EDA suggested the relationship with readmission isn't perfectly linear.

In [7]:
admissions['discharge_efficiency_bucket'] = pd.cut(
    admissions.discharge_efficiency, bins=[0,40,60,80,100],
    labels=['Rushed (0-40)','Below Avg (40-60)','Good (60-80)','Excellent (80-100)']
)
admissions['discharge_efficiency_bucket'].value_counts()

discharge_efficiency_bucket
Good (60-80)          22045
Below Avg (40-60)     11805
Excellent (80-100)    11054
Rushed (0-40)          1596
Name: count, dtype: int64

## Save the feature table

In [8]:
feature_cols = [
    'admission_id','patient_id','department_id','department','admission_type',
    'severity_score','length_of_stay_days','wait_minutes','discharge_efficiency',
    'discharge_efficiency_bucket','readmitted_30d','patient_age','age_group',
    'is_weekend_admission','admission_month','is_flu_season',
    'treatment_count','total_treatment_cost','gross_charge','margin',
    'cost_per_stay_day','satisfaction_score'
]
features = admissions[feature_cols]
features.to_csv('../data/cleaned/admissions_features.csv', index=False)
print(f"Saved feature table: {features.shape[0]:,} rows x {features.shape[1]} columns")
features.head(3)

Saved feature table: 46,500 rows x 22 columns


,admission_id,patient_id,department_id,department,admission_type,severity_score,length_of_stay_days,wait_minutes,discharge_efficiency,discharge_efficiency_bucket,...,age_group,is_weekend_admission,admission_month,is_flu_season,treatment_count,total_treatment_cost,gross_charge,margin,cost_per_stay_day,satisfaction_score
0,23509,6672,5,Pediatrics,Emergency,3.8,3.4,106.0,65.7,Good (60-80),...,0-18,False,3,False,4.0,3772.64,4107.23,2202.78,1208.01,91.0
1,26138,16754,9,Obstetrics,Urgent,3.0,7.1,15.0,66.3,Good (60-80),...,51-65,False,9,False,0.0,0.00,7965.44,2842.86,1121.89,NaN
2,18329,6959,4,Oncology,Emergency,6.3,7.9,28.0,100.0,Excellent (80-100),...,19-35,False,5,False,1.0,1495.71,15580.32,8107.10,1972.19,NaN
